# Project SQL: Data Discovery with the Chinook Database

Welcome to your first major hands-on data exploration project. In this assignment, you will be stepping into the role of a data analyst querying the Chinook database: a sample dataset representing a digital media store, complete with information about artists, albums, tracks, customers, and business sales.

The core objective of this project is to transition you from simply writing code to thinking like a data scientist. To do that, our journey follows a deliberate, four-fold progression:



### Phase 1. Master the Mechanics (Get Familiar with SQL)



Before you can extract deep insights, you need to speak the language. You will start by practicing the fundamental mechanics of SQL using the SQL statements below: learning how to filter data, join tables, group metrics, and aggregate numbers. Think of this as learning how to move around the data environment smoothly.  As a starting point, pick a single table and answer the "How many ...?" questions using SQL.  Then repeat the process for each table.



In [1]:
# Install the sqlite package for Ubuntu
# Download the Chinook sqlite database


In [2]:
#python import
import sqlite3 as db
import pandas as pd
import contextlib

In [3]:
# Install the sqlite package for Ubuntu
%%capture
%%bash
apt-get update
apt-get install -y sqlite3

In [8]:
# Download the Chinook sqlite database
%%bash
[ -f chinook.zip ] || curl -s -O https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip
unzip -o chinook.zip


Archive:  chinook.zip
  inflating: chinook.db              


In [5]:
#Remove any old copy, then unzip
!rm -f chinook.db
!unzip -u chinook.zip

Archive:  chinook.zip
  inflating: chinook.db              


In [6]:
# Check the files
!ls -l

total 1168
-rw-r--r-- 1 root root 884736 Nov 29  2015 chinook.db
-rw-r--r-- 1 root root 305596 Nov 12 14:29 chinook.zip
drwxr-xr-x 1 root root   4096 Oct 26 22:18 sample_data


### Run sql using jupyter magic

#### Count how many columns in each table

In [9]:
# List all the tables in the Chinook database
%%script sqlite3 --column --header chinook.db
.tables

albums          employees       invoices        playlists     
artists         genres          media_types     tracks        
customers       invoice_items   playlist_track


In [11]:
# Show the columns in the albums table
%%script sqlite3 --column --header chinook.db
pragma table_info(albums);

cid  name      type           notnull  dflt_value  pk
---  --------  -------------  -------  ----------  --
0    AlbumId   INTEGER        1                    1 
1    Title     NVARCHAR(160)  1                    0 
2    ArtistId  INTEGER        1                    0 


In [14]:
# How many columns are in the albums table?
%%script sqlite3 --column --header chinook.db
select count(*) as n_columns.
from pragma_table_info('albums')


Parse error near line 1: near ".": syntax error
  select count(*) as n_columns.        from pragma_table_info('albums')  
                error here ---^


CalledProcessError: Command 'b"select count(*) as n_columns.       \nfrom pragma_table_info('albums')  \n"' returned non-zero exit status 1.

In [15]:
# How many columns are in each table?
%%script sqlite3 --column --header chinook.db
select 'albums'         as table_name, count(*) as n_columns from pragma_table_info('albums');
select 'artists'        as table_name, count(*) as n_columns from pragma_table_info('artists');
select 'customers'      as table_name, count(*) as n_columns from pragma_table_info('customers');
select 'employees'      as table_name, count(*) as n_columns from pragma_table_info('employees');
select 'genres'         as table_name, count(*) as n_columns from pragma_table_info('genres');
select 'invoice_items'  as table_name, count(*) as n_columns from pragma_table_info('invoice_items');
select 'invoices'       as table_name, count(*) as n_columns from pragma_table_info('invoices');
select 'media_types'    as table_name, count(*) as n_columns from pragma_table_info('media_types');
select 'playlist_track' as table_name, count(*) as n_columns from pragma_table_info('playlist_track');
select 'playlists'      as table_name, count(*) as n_columns from pragma_table_info('playlists');
select 'tracks'         as table_name, count(*) as n_columns from pragma_table_info('tracks');

table_name  n_columns
----------  ---------
albums      3        
table_name  n_columns
----------  ---------
artists     2        
table_name  n_columns
----------  ---------
customers   13       
table_name  n_columns
----------  ---------
employees   15       
table_name  n_columns
----------  ---------
genres      2        
table_name     n_columns
-------------  ---------
invoice_items  5        
table_name  n_columns
----------  ---------
invoices    9        
table_name   n_columns
-----------  ---------
media_types  2        
table_name      n_columns
--------------  ---------
playlist_track  2        
table_name  n_columns
----------  ---------
playlists   2        
table_name  n_columns
----------  ---------
tracks      9        


#### Count how many rows in each table

In [ ]:
# How many rows are in the albums table?
%%script sqlite3 --column --header chinook.db
select count(*) as n_rows
from albums;      # look at the table of rows

In [ ]:
# How many rows are in each table?
%%script sqlite3 --column --header chinook.db
select 'albums'         as table_name, count(*) as n_rows from albums;
select 'artists'        as table_name, count(*) as n_rows from artists;
select 'customers'      as table_name, count(*) as n_rows from customers;
select 'employees'      as table_name, count(*) as n_rows from employees;
select 'genres'         as table_name, count(*) as n_rows from genres;
select 'invoice_items'  as table_name, count(*) as n_rows from invoice_items;
select 'invoices'       as table_name, count(*) as n_rows from invoices;
select 'media_types'    as table_name, count(*) as n_rows from media_types;
select 'playlist_track' as table_name, count(*) as n_rows from playlist_track;
select 'playlists'      as table_name, count(*) as n_rows from playlists;
select 'tracks'         as table_name, count(*) as n_rows from tracks;

#### Break down columns

In [16]:
# Identifier columns are the Primary keys and Foreign keys in each table
# Which column is the Primary key in "tracks"?
%%script sqlite3 --column --header chinook.db
select name, pk
from pragma_table_info('tracks')
where pk > 0;


name     pk
-------  --
TrackId  1 


In [17]:
# Which columns in track tablet is the (Foreign key) point to other tables?
%%script sqlite3 --column --header chinook.db
select "from" as column_name, "table" as points_to
from pragma_foreign_key_list('tracks');

column_name  points_to  
-----------  -----------
MediaTypeId  media_types
GenreId      genres     
AlbumId      albums     


In [18]:
# Is TrackId unique? (the two numbers should match)
%%script sqlite3 --column --header chinook.db
select count(*)                as n_rows,
       count(distinct TrackId) as n_unique_ids
from tracks;

n_rows  n_unique_ids
------  ------------
3503    3503        


### Run SQl using DuckDB

In [6]:
import duckdb
import pandas as pd

con = duckdb.connect()                                  # an empty DuckDB session in memory
con.sql("INSTALL sqlite; LOAD sqlite;")                 # add-on that lets DuckDB read SQLite files
con.sql("ATTACH 'chinook.db' AS chinook (TYPE sqlite)") # plug in your file and call it 'chinook'
con.sql("USE chinook")                                  # make it the default, so you can write 'albums'

In [12]:
#====================================================
# HOW MANY COLUMN IN EACH TABLE ?
#(note show what change if have diffrent database)
#====================================================

con.sql("""
    select table_name,                          -- never changes
           count(*) as n_columns                -- never changes (the name n_columns is your choice)
    from information_schema.columns             -- never changes
    where table_catalog = 'chinook'             -- CHANGE: your ATTACH nickname
      and table_name not like 'sqlite_%'        -- keep only for SQLite files
    group by table_name                         -- never changes
    order by table_name                         -- never changes
""").df()

,table_name,n_columns
0,albums,3
1,artists,2
2,customers,13
3,employees,15
4,genres,2
5,invoice_items,5
6,invoices,9
7,media_types,2
8,playlist_track,2
9,playlists,2


In [13]:
con.sql("SHOW TABLES").df()

,name
0,albums
1,artists
2,customers
3,employees
4,genres
5,invoice_items
6,invoices
7,media_types
8,playlist_track
9,playlists


In [14]:
# ============================================================
# HOW MANY ROWS ARE IN EACH TABLE?
# ------------------------------------------------------------
# Idea: ask "how many rows?" once per table, then stack all
#       the answers into one result table.
#
# TO REUSE ON ANOTHER DATABASE:
#   1. Run con.sql("SHOW TABLES").df() to get the table names.
#   2. Make one select line per table.
#   3. On each line, change the name in TWO places:
#        'label'      -> text shown in the output (in quotes)
#        from table   -> the real table being counted (no quotes)
#   4. Put "union all" BETWEEN lines (not before the first one,
#      not after the last one).
# ============================================================

con.sql("""
    -- Each select line answers: "How many rows are in this table?"
    --   'albums'           = a text label we type, so we know which table the number belongs to
    --   as table_name      = names the label column
    --   count(*)           = counts every row, even rows with blank values
    --   as n_rows          = names the count column
    --   from albums        = the table being counted (must match the label)

    select 'albums'         as table_name, count(*) as n_rows from albums
    union all                                   -- "stack the next answer under this one"
    select 'artists'        as table_name, count(*) as n_rows from artists
    union all
    select 'customers'      as table_name, count(*) as n_rows from customers
    union all
    select 'employees'      as table_name, count(*) as n_rows from employees
    union all
    select 'genres'         as table_name, count(*) as n_rows from genres
    union all
    select 'invoice_items'  as table_name, count(*) as n_rows from invoice_items
    union all
    select 'invoices'       as table_name, count(*) as n_rows from invoices
    union all
    select 'media_types'    as table_name, count(*) as n_rows from media_types
    union all
    select 'playlist_track' as table_name, count(*) as n_rows from playlist_track
    union all
    select 'playlists'      as table_name, count(*) as n_rows from playlists
    union all
    select 'tracks'         as table_name, count(*) as n_rows from tracks

    order by table_name                         -- sort the final stacked table A to Z (written once, at the end)
""").df()                                       # .df() only DISPLAYS the result as a pandas table; SQL did the counting

# EASY MISTAKE: changing the 'label' but not the "from" table.
#   You'd get the wrong number under the right name, and NO error.

,table_name,n_rows
0,albums,347
1,artists,275
2,customers,59
3,employees,8
4,genres,25
5,invoice_items,2240
6,invoices,412
7,media_types,5
8,playlist_track,8715
9,playlists,18


In [15]:
# ============================================================
# WHICH COLUMNS ARE PRIMARY KEYS?
# ------------------------------------------------------------
# duckdb_constraints() is DuckDB's built-in list of the "rules"
# on each table: primary keys, NOT NULL, and so on.
# We keep only the PRIMARY KEY rules.
#
# TO REUSE: change 'chinook' to your ATTACH nickname.
# ============================================================

con.sql("""
    select table_name,
           unnest(constraint_column_names) as column_name   -- unnest = split a list into one row per item
    from duckdb_constraints()                               -- built-in list of table rules
    where database_name = 'chinook'                         -- only our database
      and constraint_type = 'PRIMARY KEY'                   -- only primary-key rules
    order by table_name
""").df()

,table_name,column_name
0,albums,AlbumId
1,artists,ArtistId
2,customers,CustomerId
3,employees,EmployeeId
4,genres,GenreId
5,invoice_items,InvoiceLineId
6,invoices,InvoiceId
7,media_types,MediaTypeId
8,playlist_track,PlaylistId
9,playlist_track,TrackId


In [16]:
# ============================================================
# WHICH COLUMNS ARE FOREIGN KEYS (point to another table)?
# ------------------------------------------------------------
# DuckDB does NOT bring SQLite's foreign keys over when it
# attaches the file. So we use sqlite_query() to send a
# question straight to SQLite and get its answer back.
#
# Everything inside the single quotes '...' is SQLite's SQL.
# Inside that, '' (two single quotes) = one quote mark.
#
# TO REUSE: change 'chinook' to your ATTACH nickname.
#           (Only works for SQLite files.)
# ============================================================

con.sql("""
    select table_name, column_name
    from sqlite_query('chinook', '
        select m.name   as table_name,                  -- the table
               f."from" as column_name                  -- the column that points elsewhere
        from sqlite_master as m                          -- SQLite''s list of everything in the file
        join pragma_foreign_key_list(m.name) as f        -- foreign keys for each table
        where m.type = ''table''                         -- tables only (not indexes)
    ')
    order by table_name
""").df()

,table_name,column_name
0,albums,ArtistId
1,customers,SupportRepId
2,employees,ReportsTo
3,invoice_items,TrackId
4,invoice_items,InvoiceId
5,invoices,CustomerId
6,playlist_track,TrackId
7,playlist_track,PlaylistId
8,tracks,MediaTypeId
9,tracks,GenreId


In [19]:
# ============================================================
# HOW MANY IDENTIFIER COLUMNS (PK + FK) ARE IN EACH TABLE?
# ------------------------------------------------------------
# "with" lets us name the two lists from Cells 1 and 2,
# then use them like tables.
#
# We combine them with UNION (not UNION ALL):
#   UNION ALL = keep duplicates
#   UNION     = drop duplicates
# A column can be BOTH a primary key and a foreign key
# (playlist_track.TrackId). UNION makes sure it counts once.
#
# TO REUSE: change 'chinook' in BOTH places.
# ============================================================

con.sql("""
    with
    primary_keys as (                                   -- same as Cell 1
        select table_name,
               unnest(constraint_column_names) as column_name
        from duckdb_constraints()
        where database_name = 'chinook'
          and constraint_type = 'PRIMARY KEY'
    ),
    foreign_keys as (                                   -- same as Cell 2
        select table_name, column_name
        from sqlite_query('chinook', '
            select m.name as table_name, f."from" as column_name
            from sqlite_master as m
            join pragma_foreign_key_list(m.name) as f
            where m.type = ''table''
        ')
    ),
    identifier_columns as (                             -- stack both lists, drop repeats
        select table_name, column_name from primary_keys
        union
        select table_name, column_name from foreign_keys
    )
    select table_name,
           count(*) as n_identifier_columns            -- one row per column, so count = # of ID columns
    from identifier_columns
    group by table_name
    order by table_name
""").df()

,table_name,n_identifier_columns
0,albums,2
1,artists,1
2,customers,2
3,employees,2
4,genres,1
5,invoice_items,3
6,invoices,2
7,media_types,1
8,playlist_track,2
9,playlists,1
